# 14-day cumulative number of COVID-19 cases per 100 000 for districts (Landkreise) in Germany

At the end of the page, we provide a detailed description of how the numbers are calculated.

### Compute data

In [1]:
import datetime as dt
import pandas as pd
pd.set_option("max_rows", None)
from oscovida import get_incidence_rates_germany

## Table for all districts (=Landkreise)

In [2]:
cases_incidence, deaths_incidence = get_incidence_rates_germany()

Please be patient - downloading data from https://www.arcgis.com/sharing/rest/content/items/f10774f1c63e40168479a1feb6c7ca74/data ...
Completed downloading 201784 rows in 4.4 seconds.


In [3]:
cases_incidence

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
LK Ahrweiler,45,129727,34.7
LK Aichach-Friedberg,24,133596,18.0
LK Alb-Donau-Kreis,60,196047,30.6
LK Altenburger Land,20,90118,22.2
LK Altenkirchen,12,128705,9.3
LK Altmarkkreis Salzwedel,4,83765,4.8
LK Altötting,52,111210,46.8
LK Alzey-Worms,16,129244,12.4
LK Amberg-Sulzbach,11,103109,10.7


## Table sorted by 14-day-incidence

In [4]:
cases_incidence.sort_values(by="14-day-incidence-rate", ascending=False)

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Rosenheim,53,63324,83.7
SK Landshut,59,72404,81.5
SK München,1051,1471508,71.4
SK Heilbronn,85,125960,67.5
SK Memmingen,28,43837,63.9
SK Ulm,74,126329,58.6
SK Ingolstadt,74,136981,54.0
SK Passau,28,52469,53.4
SK Augsburg,154,295135,52.2


## Tutorial: Detailed calculation for one district

In [5]:
from oscovida import fetch_data_germany, germany_get_population
import datetime

In [6]:
period = 14 # Set the period we compute the incidence rate over

In [7]:
germany = fetch_data_germany() # Downloads population data for Germany
germany = germany.rename(
    columns={"AnzahlFall": "cases", "AnzahlTodesfall": "deaths"}
) # Rename the columns to English
germany.tail() # Print example of the table format and data

,FID,IdBundesland,Bundesland,Landkreis,Altersgruppe,Geschlecht,cases,deaths,Meldedatum,IdLandkreis,Datenstand,NeuerFall,NeuerTodesfall,Refdatum,NeuGenesen,AnzahlGenesen,IstErkrankungsbeginn,Altersgruppe2
date,,,,,,,,,,,,,,,,,,
2020-06-09,33753666,16,Thüringen,LK Altenburger Land,A80+,M,1,0,2020/06/09 00:00:00,16077,"07.09.2020, 00:00 Uhr",0,-9,2020/05/19 00:00:00,0,1,1,Nicht übermittelt
2020-05-06,33753667,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/06 00:00:00,16077,"07.09.2020, 00:00 Uhr",0,-9,2020/05/04 00:00:00,0,1,1,Nicht übermittelt
2020-05-11,33753668,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/11 00:00:00,16077,"07.09.2020, 00:00 Uhr",0,-9,2020/05/01 00:00:00,0,1,1,Nicht übermittelt
2020-05-28,33753669,16,Thüringen,LK Altenburger Land,A80+,W,2,0,2020/05/28 00:00:00,16077,"07.09.2020, 00:00 Uhr",0,-9,2020/05/28 00:00:00,0,2,0,Nicht übermittelt
2020-06-09,33753670,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/06/09 00:00:00,16077,"07.09.2020, 00:00 Uhr",0,-9,2020/06/09 00:00:00,0,1,0,Nicht übermittelt


In [8]:
yesterday = datetime.datetime.now() - datetime.timedelta(days=1)
fortnight_ago = yesterday - datetime.timedelta(days=period)
periods = (fortnight_ago < germany.index) & (germany.index < yesterday) # Create a period mask
germany = germany.iloc[periods]

In [9]:
cases_germany = germany[["Landkreis", "cases"]]
cases_germany.tail() # The data for germany has the date as the index, and the location data on each column

,Landkreis,cases
date,,
2020-08-28,LK Altenburger Land,3
2020-09-02,LK Altenburger Land,1
2020-08-25,LK Altenburger Land,1
2020-08-28,LK Altenburger Land,1
2020-09-04,LK Altenburger Land,1


In [10]:
cases_sum = (
    cases_germany.groupby("Landkreis")
    .sum()
    .rename(columns={"cases": f"{period}-day-sum"})
) # Here we sum the daily data to get total data
cases_sum.tail()

,14-day-sum
Landkreis,
SK Worms,13
SK Wuppertal,99
SK Würzburg,26
SK Zweibrücken,3
StadtRegion Aachen,79


In [11]:
population = (
    germany_get_population()
    .population
    .to_frame()
) # This gives us population per county, with county as the index
population.tail()

,population
county,
SK Berlin Spandau,243977
SK Berlin Steglitz-Zehlendorf,308697
SK Berlin Mitte,384172
SK Berlin Friedrichshain-Kreuzberg,289762
SK Berlin Tempelhof-Schöneberg,351644


In [12]:
cases_incidence = cases_sum.join(population)
cases_incidence[f"{period}-day-incidence-rate"] = (
    cases_incidence[f"{period}-day-sum"] / cases_incidence["population"] * 100_000
) # By convention the incedence rate is the total cases in the past x days / population * 100_000

In [13]:
cases_incidence.tail()

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Worms,13,83330,15.600624
SK Wuppertal,99,354382,27.935956
SK Würzburg,26,127880,20.331561
SK Zweibrücken,3,34209,8.769622
StadtRegion Aachen,79,555465,14.222318


In [14]:
cases_incidence.loc['SK Hamburg']

14-day-sum               3.860000e+02
population               1.841179e+06
14-day-incidence-rate    2.096483e+01
Name: SK Hamburg, dtype: float64